In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rdkit import Chem
from rdkit import RDLogger
RDLogger.logger().setLevel(RDLogger.CRITICAL)

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def canon(s: str):
    if not s or not s.strip():
        return None
    try:
        m = Chem.MolFromSmiles(s)
        if m is None:
            return None
        return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    except Exception:
        return None

In [3]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("sagawa/CompoundT5")
model = AutoModelForSeq2SeqLM.from_pretrained("sagawa/CompoundT5").to(device)
model.eval()
model.config.use_cache = True

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
test_enc = torch.load("../tokenizer/pretraining/test_enc.pt", weights_only=False)

labels_all = test_enc["labels"].clone()
labels_all[labels_all == -100] = tokenizer.pad_token_id
gold_all = tokenizer.batch_decode(labels_all, skip_special_tokens=True)
src_all = tokenizer.batch_decode(test_enc["input_ids"], skip_special_tokens=True)


bs = 32
topk = 5
N = len(gold_all)
rows = []
for i in range(0, N, bs):
    input_ids      = test_enc["input_ids"][i:i+bs].to(device)
    attention_mask = test_enc["attention_mask"][i:i+bs].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=topk,
            num_return_sequences=topk,
            do_sample=False,
            early_stopping=True,
        )

    decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    for b in range(i, min(i + bs, N)):
        offset = (b - i) * topk
        preds = decoded[offset:offset + topk]

        row = {
            "idx": b,
            "src": src_all[b].strip(),
            "gold": gold_all[b].strip(),
        }
        for k in range(topk):
            row[f"pred_{k+1}"] = preds[k].strip() if k < len(preds) else ""
        row["pred_text_all"] = " || ".join(p.strip() for p in preds)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("results.csv", index=False)
df

,idx,src,gold,pred_1,pred_2,pred_3,pred_4,pred_5,pred_text_all
0,0,CCC(C)S(C)(=O)=OOC=OO=[N+]([O-])c1ccccc1Sc1ccc...,CS(=O)(=O)C(COC(=O)c1ccc([N+](=O)[O-])cc1)C(CS...,[C@H]()C(C)(C)N(C,[C@H]()N(CC(C)C,[C@H]()N(C,[C@H](,,[C@H]()C(C)(C)N(C || [C@H]()N(CC(C)C || [C@H](...
1,1,Cc1ccc2ccccc2n1NC=OCc1ccccc1N,Cc1cc(NC(=O)c2cccc(C)c2N)c2ccccc2n1,c1cc(N)ccc1,c1ccc(N)cc1,,c1ccc(N)cc1,C(C)(C),c1cc(N)ccc1 || c1ccc(N)cc1 || || c1ccc(N)cc1 ...
2,2,CCOCC(C)(C)CCC=OO=C1CCN2CCCN12O=[N+]([O-])c1cc...,CCOC(=O)C(CC(C)(C)C)=C1CC(c2ccc([N+](=O)[O-])c...,,C(O),),C,O,|| C(O) || ) || C || O
3,3,CCC=ONCC(C)(C)CC=OO=C1NCc2ccccc21NCC(C)CC(=O)C...,CC(NC(=O)C(N1Cc2ccccc2C1=O)C(C)(C)C)C(=O)NC(C(...,C(=O),C(CC),CC,C,,C(=O) || C(CC) || CC || C ||
4,4,Cc1ccccc1CCNO=Cn1ccc(=O)o1c1ccccc1,Cc1ccc(CNC(=O)n2oc(=O)cc2c2ccccc2)cc1C,(C)(C),(C),=,(O)C(O),(O),(C)(C) || (C) || = || (O)C(O) || (O)
...,...,...,...,...,...,...,...,...,...
49995,49995,CCCCCCCC[NH2+]CC(O)CCCCCCCCCO,CCCCCCCC[NH2+]C(O)CCCCCCCCC(C)O,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...,CC[C@@H](O)CC)CC[NH2+]C[C@@H](O)C[C@@H](O)CCCC...
49996,49996,CC(C)C(C)CNC=OOCC(C)C[NH2+]CCc1cncn1,CC(C)C(CNC(=O)OC(C)(C)C)C[NH2+]CCn1ccnc1,C(C)CC(C)CC(CC(C)CC(C(C)C)CC(C)C,C(C)CC(C)CC(CC(C),C(C)CC(C)CC(CC(C)CC(C)C(C)C,C(C)C(C)C(C)CC(CC(C),C(C)C(C)CC(C)CC(CC(C),C(C)CC(C)CC(CC(C)CC(C(C)C)CC(C)C || C(C)CC(C)C...
49997,49997,Cc1ccc(Br)cc1C=OCC1(C)CNc2ccccc21,Cc1ccc(Br)cc1C(=O)N1CC(C)(C)c2ccccc21,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...,c2ccc1c1ccc2c(c1)-c1c1ccc2c(c1)Cc1ccc2ccc3cccc...
49998,49998,CCCCS(=O)(=O)C1CC(=O)N1CC=OOCC(C)CCC=OOCC(C)C,CC(C)=C(C(=O)OC(C)(C)C)N1C(=O)C(=CC(=O)OC(C)(C...,C(CC)C(C)C,C(C)C(CC)CC,C(C)C(C)C,(C)C(CC)CC,,C(CC)C(C)C || C(C)C(CC)CC || C(C)C(C)C || (C)C...
